# 📉 Churn Demo — Two Predictions in One Dataset
### `Outcome = f(Problem, Data, Model, Evaluation, Explainability)`

Most teams stop at: *"Will this customer churn?"* (classification)  
The better question is: *"When will this customer churn?"* (survival / time-to-event)

This notebook shows both, on the same synthetic bank customer dataset:

| Recipe | Question | Model | Primary Metric |
|--------|----------|-------|---------------|
| **Classify** | Will this customer churn in 90 days? | XGBoost | PR-AUC |
| **Predict** | How many days until they churn? | Survival (Kaplan-Meier + Cox) | C-index / Expected Survival |

**Install once if needed:**
```bash
pip install faker lifelines xgboost shap scikit-learn pandas numpy matplotlib seaborn
```

In [1]:
# ── IMPORTS ────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from faker import Faker
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve
import xgboost as xgb
import shap

# Survival analysis
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.utils import concordance_index

# Palette matching the deck
C = dict(
    navy='#1E2D4E', navyDk='#141D30', teal='#028090', mint='#02C39A',
    purple='#5B2C8D', orange='#BF5E00', red='#C0392B',
    light='#EFF4F8', muted='#8899AA', white='#FFFFFF'
)
DARK_THEME = {
    'figure.facecolor': C['navyDk'],
    'axes.facecolor':   C['navy'],
    'axes.edgecolor':   '#3A4F70',
    'axes.labelcolor':  '#CCDDEE',
    'xtick.color':      '#8899AA',
    'ytick.color':      '#8899AA',
    'text.color':       '#CCDDEE',
    'grid.color':       '#2A3D5A',
    'grid.alpha':       0.5,
    'font.family':      'sans-serif',
}
plt.rcParams.update(DARK_THEME)

SEED = 42
np.random.seed(SEED)
print('✅  Libraries loaded')

ModuleNotFoundError: No module named 'numpy'

---
## Step 1 — Generate Synthetic Customer Data (Faker)

We simulate a bank's customer base: 20,000 customers observed over up to 2 years.  
Some have churned (event observed). Others are still active (right-censored — we know they haven't churned *yet* but don't know when they will).  

> **Right-censoring** is the key survival analysis concept. Standard regression ignores it. Survival models don't.

In [ ]:
# ── GENERATE SYNTHETIC CUSTOMER DATA ──────────────────────────────────────────
fake = Faker()
Faker.seed(SEED)
rng  = np.random.default_rng(SEED)
N    = 20_000

# ── Customer attributes ──
tenure_days      = rng.integers(30, 730, N)           # how long they've been a customer
age              = rng.integers(22, 75, N)
products_held    = rng.integers(1, 6, N)               # checking, savings, credit, mortgage, invest
monthly_balance  = np.abs(rng.normal(5000, 4000, N)).clip(50, 80000).round(2)
login_freq_30d   = rng.integers(0, 45, N)              # logins in last 30 days
support_tickets  = rng.integers(0, 8, N)               # complaints in last 90 days
mobile_app_user  = rng.integers(0, 2, N)               # 0=no, 1=yes
nps_score        = rng.integers(1, 11, N)              # 1-10 NPS
rate_competitive = rng.uniform(0.5, 1.5, N).round(3)   # 1.0 = at market rate

segments = ['Mass Market', 'Emerging Affluent', 'Affluent', 'Small Business', 'Student']
segment  = rng.choice(segments, N, p=[0.40, 0.25, 0.15, 0.12, 0.08])

# ── Build churn hazard (latent) ──
# More likely to churn if: low logins, low products, high support tickets, low NPS, above-market rates
hazard = (
    0.30 * (1 - login_freq_30d / 45)
  + 0.20 * (1 - products_held / 5)
  + 0.15 * (support_tickets / 8)
  + 0.20 * (1 - nps_score / 10)
  + 0.15 * (rate_competitive - 0.5) / 1.0
  + rng.normal(0, 0.05, N)  # noise
).clip(0, 1)

# Student and Mass Market have higher base churn rate
hazard += np.where(segment == 'Student', 0.12, 0)
hazard += np.where(segment == 'Mass Market', 0.05, 0)
hazard  = hazard.clip(0, 0.95)

# ── Simulate time-to-churn (Weibull-ish) ──
# Scale hazard to time: higher hazard → shorter survival
# Observed window: 730 days
WINDOW = 730
# Expected survival time from hazard (inverse relationship)
scale_param  = (1 - hazard) * WINDOW * 0.9 + 30   # days until churn on average
time_to_churn = rng.exponential(scale=scale_param).clip(1, WINDOW * 2).round().astype(int)

# Censoring: if churn time > observation window, we don't observe the event
observed_time = np.minimum(time_to_churn, WINDOW)
churned       = (time_to_churn <= WINDOW).astype(int)

# Binary 90-day churn label
churn_90d     = (time_to_churn <= 90).astype(int)

df = pd.DataFrame({
    'customer_id':     [f'C{rng.integers(10000,99999)}' for _ in range(N)],
    'tenure_days':     tenure_days,
    'age':             age,
    'products_held':   products_held,
    'monthly_balance': monthly_balance,
    'login_freq_30d':  login_freq_30d,
    'support_tickets': support_tickets,
    'mobile_app_user': mobile_app_user,
    'nps_score':       nps_score,
    'rate_competitive':rate_competitive,
    'segment':         segment,
    # Survival analysis targets
    'observed_days':   observed_time,   # how long we observed this customer
    'churned':         churned,          # 1=churned within window, 0=still active (censored)
    # Classification target
    'churn_90d':       churn_90d,
})

print(f'Customers:          {N:,}')
print(f'Churned (2yr):      {churned.sum():,}  ({churned.mean()*100:.1f}%)')
print(f'Still active:       {(1-churned).sum():,}  ({(1-churned).mean()*100:.1f}%)  ← right-censored')
print(f'Churn in 90d:       {churn_90d.sum():,}  ({churn_90d.mean()*100:.1f}%)')
df.head(5)

---
## Step 2 — EDA: Who's Leaving and Why?

> *"Before models — understand the data. The detective work starts here."*

In [ ]:
# ── EXPLORATORY DATA ANALYSIS ─────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('EDA — Who Churns and Why?', fontsize=13, fontweight='bold',
             color='white', y=1.01)

# 1. Churn rate by segment
ax = axes[0, 0]
seg_churn = df.groupby('segment')['churned'].mean().sort_values(ascending=True)
colors_seg = [C['red'] if v > 0.55 else C['teal'] for v in seg_churn.values]
ax.barh(seg_churn.index, seg_churn.values * 100, color=colors_seg, height=0.55)
ax.set_xlabel('2-Year Churn Rate (%)')
ax.set_title('Churn Rate by Segment', color='white')
for i, v in enumerate(seg_churn.values):
    ax.text(v*100 + 0.5, i, f'{v*100:.1f}%', va='center', fontsize=8.5, color='white')

# 2. Login frequency distribution
ax = axes[0, 1]
churned_logins = df.loc[df['churned']==1, 'login_freq_30d']
active_logins  = df.loc[df['churned']==0, 'login_freq_30d']
ax.hist(active_logins,  bins=30, density=True, alpha=0.6, color=C['teal'],   label='Active')
ax.hist(churned_logins, bins=30, density=True, alpha=0.7, color=C['red'],    label='Churned')
ax.set_xlabel('Logins in last 30 days')
ax.set_ylabel('Density')
ax.set_title('Login Frequency by Churn Status', color='white')
ax.legend(fontsize=9)

# 3. Products held vs churn rate
ax = axes[0, 2]
prod_churn = df.groupby('products_held')['churned'].mean()
ax.bar(prod_churn.index, prod_churn.values*100, color=C['purple'], width=0.6, edgecolor='none')
ax.set_xlabel('# Products Held')
ax.set_ylabel('Churn Rate (%)')
ax.set_title('Products Held vs Churn Rate', color='white')
ax.set_xticks(range(1, 6))

# 4. NPS vs churn
ax = axes[1, 0]
nps_churn = df.groupby('nps_score')['churned'].mean()
ax.bar(nps_churn.index, nps_churn.values*100,
       color=[C['red'] if v > 0.55 else C['orange'] if v > 0.45 else C['mint']
              for v in nps_churn.values], width=0.7, edgecolor='none')
ax.set_xlabel('NPS Score (1–10)')
ax.set_ylabel('Churn Rate (%)')
ax.set_title('NPS Score vs Churn Rate', color='white')

# 5. Survival curve preview
ax = axes[1, 1]
kmf = KaplanMeierFitter()
kmf.fit(df['observed_days'], event_observed=df['churned'], label='All Customers')
kmf.plot_survival_function(ax=ax, color=C['mint'], ci_alpha=0.15)
ax.set_xlabel('Days Since Observation Start')
ax.set_ylabel('Survival Probability (not churned)')
ax.set_title('Kaplan-Meier Survival Curve', color='white')
ax.set_ylim(0, 1)
ax.legend(fontsize=9)

# 6. Time to churn distribution (observed churners)
ax = axes[1, 2]
churner_times = df.loc[df['churned']==1, 'observed_days']
ax.hist(churner_times, bins=40, color=C['orange'], edgecolor='none', alpha=0.85)
ax.axvline(90,  color=C['red'],  linestyle='--', linewidth=1.5, label='90-day window')
ax.axvline(365, color=C['teal'], linestyle='--', linewidth=1.5, label='1-year window')
ax.set_xlabel('Days Until Churn (observed)')
ax.set_ylabel('Count')
ax.set_title('Time-to-Churn Distribution', color='white')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

---
## Part A — Classification: Will This Customer Churn in 90 Days?

> *"You're choosing a type of question, not an algorithm."*  
> Here the question is binary: churn or not in the next 90 days → **Classify recipe**.

In [ ]:
# ── FEATURE PREP ──────────────────────────────────────────────────────────────
le = LabelEncoder()
df['segment_enc'] = le.fit_transform(df['segment'])

FEATURES = [
    'tenure_days', 'age', 'products_held', 'monthly_balance',
    'login_freq_30d', 'support_tickets', 'mobile_app_user',
    'nps_score', 'rate_competitive', 'segment_enc'
]

X = df[FEATURES]
y_clf = df['churn_90d']

X_train, X_test, y_train, y_test = train_test_split(
    X, y_clf, test_size=0.25, stratify=y_clf, random_state=SEED
)

# ── TRAIN XGBoost ──
pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
clf = xgb.XGBClassifier(
    n_estimators=400,
    max_depth=5,
    learning_rate=0.04,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=8,
    scale_pos_weight=pos_weight,
    eval_metric='aucpr',
    random_state=SEED,
    verbosity=0
)
clf.fit(X_train, y_train)
clf_proba = clf.predict_proba(X_test)[:, 1]

pr_auc = average_precision_score(y_test, clf_proba)
roc    = roc_auc_score(y_test, clf_proba)
print(f'Classification Model — 90-Day Churn')
print(f'  PR-AUC:  {pr_auc:.4f}')
print(f'  ROC-AUC: {roc:.4f}')
print(f'  Churn rate in test: {y_test.mean()*100:.1f}%')

In [ ]:
# ── CLASSIFICATION EVALUATION ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
fig.suptitle('Part A — Classification: 90-Day Churn Prediction',
             fontsize=12, fontweight='bold', color='white')

# PR Curve
ax = axes[0]
p, r, thresh = precision_recall_curve(y_test, clf_proba)
f1 = np.where((p+r)>0, 2*p*r/(p+r), 0)
best_idx = np.argmax(f1[:-1])
ax.plot(r, p, color=C['teal'], linewidth=2.5, label=f'XGBoost PR-AUC={pr_auc:.3f}')
ax.axhline(y_test.mean(), color=C['muted'], linestyle='--', linewidth=1,
           label=f'Baseline ({y_test.mean():.3f})')
ax.scatter(r[best_idx], p[best_idx], color=C['mint'], s=80, zorder=5,
           label=f'Best F1 θ={thresh[best_idx]:.2f}')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve', color='white')
ax.legend(fontsize=8); ax.set_xlim(0,1); ax.set_ylim(0,1)

# Score distribution
ax = axes[1]
ax.hist(clf_proba[y_test==0], bins=50, density=True, alpha=0.6,
        color=C['teal'], label='Retained')
ax.hist(clf_proba[y_test==1], bins=50, density=True, alpha=0.75,
        color=C['red'], label='Churned')
ax.axvline(thresh[best_idx], color='white', linestyle=':', linewidth=1.5)
ax.set_xlabel('Predicted Churn Probability')
ax.set_ylabel('Density')
ax.set_title('Score Separation', color='white')
ax.legend(fontsize=9)

# Business impact: intervention curve
ax = axes[2]
INTERVENTION_COST = 50   # $50 retention offer
CHURN_LOSS        = 800  # $800 avg annual value lost if churn
thresholds = np.linspace(0.01, 0.99, 100)
net_values = []
for t in thresholds:
    predicted_churn = (clf_proba >= t)
    tp = ((predicted_churn) & (y_test==1)).sum()  # caught churners
    fp = ((predicted_churn) & (y_test==0)).sum()  # wasted interventions
    # Assume 40% save rate on correctly identified churners
    net_values.append(tp * 0.4 * CHURN_LOSS - fp * INTERVENTION_COST - tp * INTERVENTION_COST)
best_biz_idx = np.argmax(net_values)
ax.plot(thresholds, net_values, color=C['mint'], linewidth=2)
ax.axvline(thresholds[best_biz_idx], color=C['orange'], linestyle='--', linewidth=1.5,
           label=f'Max value θ={thresholds[best_biz_idx]:.2f}')
ax.axhline(0, color=C['muted'], linewidth=0.8)
ax.set_xlabel('Decision Threshold')
ax.set_ylabel('Net Value ($)')
ax.set_title(f'Business Value  (Intv=\${INTERVENTION_COST} | 40% save | Loss=\${CHURN_LOSS})',
             color='white', fontsize=9)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x:,.0f}'))
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

print(f'\n💡 Optimal intervention threshold: {thresholds[best_biz_idx]:.2f}')
print(f'   Send retention offer to customers with churn score > {thresholds[best_biz_idx]:.2f}')
print(f'   Expected net value: ${net_values[best_biz_idx]:,.0f} on test set')

---
## Part B — Survival Analysis: WHEN Will They Churn?

> *"Will they churn" tells you who to call. "When will they churn" tells you how urgently to call them.*

### Why standard regression fails here:

- **Right-censoring**: 45% of customers haven't churned yet. Their `observed_days` is a *lower bound* on their true survival time — not the actual value. Regression treats it as the actual value and is systematically biased.
- **Survival models handle censoring correctly** by using the likelihood of having survived *at least* that long.

### Two approaches:
1. **Kaplan-Meier** — non-parametric, descriptive. "What fraction of customers survive to day X?"
2. **Cox Proportional Hazards** — semi-parametric regression. "How do features affect the hazard rate?"

In [ ]:
# ── KAPLAN-MEIER SURVIVAL CURVES BY SEGMENT ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
fig.suptitle('Part B — Survival Analysis: When Do Customers Churn?',
             fontsize=12, fontweight='bold', color='white')

seg_colors = {
    'Mass Market':       C['red'],
    'Student':           C['orange'],
    'Small Business':    C['purple'],
    'Emerging Affluent': C['teal'],
    'Affluent':          C['mint'],
}

ax = axes[0]
kmf_segs = {}
for seg in segments:
    mask  = df['segment'] == seg
    kmf_s = KaplanMeierFitter()
    kmf_s.fit(df.loc[mask, 'observed_days'],
              event_observed=df.loc[mask, 'churned'],
              label=seg)
    kmf_s.plot_survival_function(ax=ax, color=seg_colors[seg], ci_show=False)
    kmf_segs[seg] = kmf_s

ax.set_xlabel('Days Since Observation Start')
ax.set_ylabel('Probability of NOT Having Churned')
ax.set_title('Survival by Segment\n(lower = higher churn risk)', color='white')
ax.set_ylim(0, 1)
ax.legend(fontsize=8.5)

# Median survival time by segment
ax = axes[1]
median_survivals = {}
for seg, kmf_s in kmf_segs.items():
    med = kmf_s.median_survival_time_
    median_survivals[seg] = med if med < np.inf else 730

seg_order = sorted(median_survivals, key=median_survivals.get)
vals      = [median_survivals[s] for s in seg_order]
colors_ms = [seg_colors[s] for s in seg_order]
bars = ax.barh(seg_order, vals, color=colors_ms, height=0.55, edgecolor='none')
ax.axvline(90,  color=C['red'],  linestyle='--', linewidth=1.2, alpha=0.8, label='90 days')
ax.axvline(365, color=C['muted'],linestyle='--', linewidth=1.0, alpha=0.7, label='1 year')
for bar, val in zip(bars, vals):
    label = f'{val:.0f}d' if val < 730 else '>730d'
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
            label, va='center', fontsize=9, color='white')
ax.set_xlabel('Median Survival Time (days)')
ax.set_title('Median Time Until Churn by Segment', color='white')
ax.legend(fontsize=8.5)

plt.tight_layout()
plt.show()

print('\nMedian Survival Time by Segment:')
for seg in seg_order:
    v = median_survivals[seg]
    print(f'  {seg:<22}: {v:.0f} days  ({v/365:.1f} years)')

In [ ]:
# ── COX PROPORTIONAL HAZARDS MODEL ────────────────────────────────────────────
# Cox regression: how do features affect the hazard (churn rate)?
# Hazard ratio > 1 → feature increases churn risk
# Hazard ratio < 1 → feature decreases churn risk

cox_df = df[FEATURES + ['observed_days', 'churned']].copy()

# Normalize continuous features for Cox (helps convergence)
for col in ['tenure_days','monthly_balance','login_freq_30d','rate_competitive']:
    cox_df[col] = (cox_df[col] - cox_df[col].mean()) / cox_df[col].std()

cph = CoxPHFitter(penalizer=0.1)
cph.fit(cox_df, duration_col='observed_days', event_col='churned',
        show_progress=False)

print('Cox Proportional Hazards — Summary:')
cph.print_summary(decimals=3, model='proportional hazard')

print(f'\n  Concordance Index (C-index): {cph.concordance_index_:.4f}')
print(f'  (0.5 = random, 1.0 = perfect ranking of churn timing)')

In [ ]:
# ── COX HAZARD RATIOS VISUALIZATION ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
fig.suptitle('Cox Model — What Drives Churn Timing?',
             fontsize=12, fontweight='bold', color='white')

# Hazard ratios (exp(coef)) with confidence intervals
ax = axes[0]
summary = cph.summary[['exp(coef)', 'exp(coef) lower 95%', 'exp(coef) upper 95%', 'p']].copy()
summary = summary.sort_values('exp(coef)')
feat_labels = summary.index.tolist()
hr     = summary['exp(coef)'].values
hr_lo  = summary['exp(coef) lower 95%'].values
hr_hi  = summary['exp(coef) upper 95%'].values
pvals  = summary['p'].values

colors_hr = [C['red'] if h > 1.0 else C['mint'] for h in hr]
y_pos = range(len(feat_labels))

ax.barh(y_pos, hr - 1, left=1, color=colors_hr, height=0.55, alpha=0.8, edgecolor='none')
ax.errorbar(hr, y_pos,
            xerr=[hr - hr_lo, hr_hi - hr],
            fmt='none', color='white', linewidth=1.2, capsize=4)
ax.axvline(1.0, color='white', linewidth=1.5, linestyle='-')
ax.set_yticks(y_pos)
ax.set_yticklabels(feat_labels, fontsize=9)
ax.set_xlabel('Hazard Ratio (HR > 1 → more churn risk)')
ax.set_title('Hazard Ratios with 95% CI', color='white')

for i, (h, p) in enumerate(zip(hr, pvals)):
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
    ax.text(max(hr_hi[i], 1.02) + 0.02, i, f'{h:.2f} {sig}',
            va='center', fontsize=7.5, color='white')

red_p   = mpatches.Patch(color=C['red'],  label='HR > 1 (increases churn risk)')
green_p = mpatches.Patch(color=C['mint'], label='HR < 1 (decreases churn risk)')
ax.legend(handles=[red_p, green_p], fontsize=8,
          facecolor='#2A3D5A', edgecolor='none', labelcolor='white')

# Individual survival curves for selected customers
ax = axes[1]
# Pick 4 representative customers: high-risk and low-risk
df['churn_score'] = clf.predict_proba(X)[:, 1]
high_risk = df.nlargest(3, 'churn_score').index[:3]
low_risk  = df.nsmallest(3, 'churn_score').index[:3]

t_range = np.linspace(0, 730, 200)

for i, idx in enumerate(high_risk):
    row = cox_df.loc[[idx], FEATURES]
    sf  = cph.predict_survival_function(row, times=t_range)
    ax.plot(t_range, sf.values.flatten(), color=C['red'],
            alpha=0.7 - i*0.1, linewidth=1.8,
            label='High-risk customer' if i==0 else '')

for i, idx in enumerate(low_risk):
    row = cox_df.loc[[idx], FEATURES]
    sf  = cph.predict_survival_function(row, times=t_range)
    ax.plot(t_range, sf.values.flatten(), color=C['mint'],
            alpha=0.7 - i*0.1, linewidth=1.8,
            label='Low-risk customer' if i==0 else '')

ax.axvline(90,  color=C['orange'], linestyle='--', linewidth=1.2, label='90-day horizon')
ax.set_xlabel('Days')
ax.set_ylabel('P(Still a customer)')
ax.set_title('Individual Survival Curves\n(Cox model predictions per customer)',
             color='white', fontsize=10)
ax.set_ylim(0, 1)
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

---
## Part C — SHAP Explainability on the Classifier

> *"Why is THIS customer high-risk?"* — Local explainability for the retention team.

In [ ]:
# ── SHAP: GLOBAL + LOCAL ───────────────────────────────────────────────────────
print('Computing SHAP values...')
explainer = shap.TreeExplainer(clf)
shap_vals  = explainer.shap_values(X_test)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
fig.suptitle('Explainability — What Drives Churn Risk?',
             fontsize=12, fontweight='bold', color='white')

# Global: mean |SHAP|
ax = axes[0]
mean_shap = np.abs(shap_vals).mean(axis=0)
order     = np.argsort(mean_shap)
bar_colors_shap = [C['mint'] if v == mean_shap.max() else C['teal']
                   for v in mean_shap[order]]
ax.barh([FEATURES[i] for i in order], mean_shap[order],
        color=bar_colors_shap, height=0.6, edgecolor='none')
ax.set_xlabel('Mean |SHAP value| — Impact on Churn Score')
ax.set_title('Global Feature Importance', color='white')

# Local: waterfall for highest-risk customer
ax = axes[1]
test_scores = clf.predict_proba(X_test)[:, 1]
worst_idx_local = np.argmax(test_scores)
worst_sv  = shap_vals[worst_idx_local]
worst_row = X_test.iloc[worst_idx_local]
worst_score_val = test_scores[worst_idx_local]

feat_sv   = sorted(zip(FEATURES, worst_sv), key=lambda x: abs(x[1]), reverse=True)
names_sv  = [f[0] for f in feat_sv]
vals_sv   = [f[1] for f in feat_sv]
colors_sv = [C['red'] if v > 0 else C['mint'] for v in vals_sv]

bars_sv = ax.barh(names_sv[::-1], [abs(v) for v in vals_sv[::-1]],
                  color=[C['red'] if v > 0 else C['mint'] for v in vals_sv[::-1]],
                  height=0.55, edgecolor='none')
for bar, (feat, sv) in zip(bars_sv, zip(names_sv[::-1], vals_sv[::-1])):
    fv  = worst_row[feat]
    sign = '+' if sv > 0 else '−'
    ax.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height()/2,
            f'{sign}{abs(sv):.3f}  (val={fv:.1f})',
            va='center', ha='left', fontsize=8, color='white')

ax.set_xlabel('|SHAP contribution to churn score|')
ax.set_title(f'Local Explanation — Highest Risk Customer\nChurn Score: {worst_score_val:.3f}',
             color='white')

red_p   = mpatches.Patch(color=C['red'],  label='Increases churn risk')
green_p = mpatches.Patch(color=C['mint'], label='Decreases churn risk')
ax.legend(handles=[red_p, green_p], fontsize=8.5,
          facecolor='#2A3D5A', edgecolor='none', labelcolor='white')

plt.tight_layout()
plt.show()

# Operational explanation
top3_risk = [(n, v) for n, v in zip(names_sv, vals_sv) if v > 0][:3]
top1_save = [(n, v) for n, v in zip(names_sv, vals_sv) if v < 0][:1]

human_map = {
    'login_freq_30d':   lambda v: f'only {int(v)} logins in the last 30 days',
    'nps_score':        lambda v: f'NPS score of {int(v)}/10 — dissatisfied',
    'products_held':    lambda v: f'only {int(v)} product(s) — low stickiness',
    'support_tickets':  lambda v: f'{int(v)} support tickets in 90 days',
    'monthly_balance':  lambda v: f'balance ${v:,.0f}',
    'rate_competitive': lambda v: f'rate {v:.2f}x market rate',
    'tenure_days':      lambda v: f'tenure of {int(v)} days',
    'age':              lambda v: f'age {int(v)}',
    'mobile_app_user':  lambda v: 'no mobile app' if v < 0.5 else 'mobile app user',
    'segment_enc':      lambda v: f'segment code {int(v)}',
}

print(f'\n📞 Retention Team Alert — Customer {X_test.index[worst_idx_local]}:')
print(f'   Churn probability (90d): {worst_score_val*100:.1f}%')
print(f'   Risk factors:')
for feat, sv in top3_risk:
    fv = worst_row[feat]
    desc = human_map.get(feat, lambda v: f'{feat}={v:.2f}')(fv)
    print(f'   → {desc}')
if top1_save:
    feat, sv = top1_save[0]
    fv = worst_row[feat]
    desc = human_map.get(feat, lambda v: f'{feat}={v:.2f}')(fv)
    print(f'   Protective factor: {desc}')

---
## Summary — Two Models, One Dataset

| Model | Question | Output | Use Case |
|-------|----------|--------|----------|
| **XGBoost Classifier** | Will they churn in 90 days? | Probability 0–1 | Trigger intervention: who to call today |
| **Kaplan-Meier** | What fraction survives to day X? | Survival curve | Segment strategy, lifetime value planning |
| **Cox PH** | How do features affect churn timing? | Hazard ratios | Feature understanding, product decisions |
| **SHAP Local** | Why is THIS customer high-risk? | Feature contributions | Personalized retention scripts |

**The combo:**
1. Run **Cox model** to understand what drives churn timing across your portfolio
2. Use **XGBoost score** to rank customers for daily outreach
3. Use **SHAP waterfall** to give the retention rep a reason-specific script
4. Use **Kaplan-Meier by segment** to prioritize which segments need structural product changes

> *"Will they churn tells you who to call. When they churn tells you how urgently. Why they churn tells you what to say."*